In [ ]:
import pandas as pd
df = pd.read_csv('all-data.csv',encoding='iso-8859-1',names=['label','text'])

In [ ]:
from sklearn.model_selection import train_test_split

x, x_test, y, y_test = train_test_split(df.text,df.label,test_size=0.2,train_size=0.8,random_state=100)
x_train, x_cv, y_train, y_cv = train_test_split(x,y,test_size = 0.25,train_size =0.75,random_state=100)

In [ ]:
df_train = pd.DataFrame({'text': x_train.values,'label': y_train.values})
df_train.to_csv('train.csv')

df_test = pd.DataFrame({'text': x_test.values,'label': y_test.values})
df_test.to_csv('test.csv')

df_val = pd.DataFrame({'text': x_cv.values,'label': y_cv.values})
df_val.to_csv('val.csv')

df_test

In [ ]:
df_test['Neutral'] = -1.0
df_test['Negative'] = -1.0
df_test['Positive'] = -1.0
df_test['FinBERT'] = "nan"
df_test

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline


model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

pipe = TextClassificationPipeline(
model=model,top_k=3,
tokenizer=tokenizer,
return_all_scores=True, # get scores for all labels
function_to_apply="softmax" # convert logits to probabilities
)

In [ ]:
for t in df_test['text']:
    scores = pipe(t)[0]
    df_test.loc[df_test['text']==t,'Neutral'] = scores[0]['score']
    df_test.loc[df_test['text']==t,'Positive'] = scores[1]['score']
    df_test.loc[df_test['text']==t,'Negative'] = scores[2]['score']
    top = max(scores, key=lambda x: x["score"])
    df_test.loc[df_test['text']==t,'FinBERT'] = top['label'].lower()


print(df_test)

In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(df_test.label, df_test.FinBERT))

In [ ]:
from datasets import load_dataset
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
import numpy as np
import evaluate
import os

model_name = "yiyanghkust/finbert-tone"
train_path = "train.csv"
val_path = "val.csv"
test_path = "test.csv"
num_labels = 3
max_length = None
data_files = {"train": train_path, "validation": val_path, "test": test_path}
raw = load_dataset("csv", data_files=data_files)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
   
def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

tokenized = raw.map(preprocess, batched=True, remove_columns=[c for c in raw["train"].column_names if c not in ("text","label")])

In [ ]:
label2id = {"neutral":0, "positive":1, "negative":2}
tokenized = tokenized.map(lambda x: {"label": label2id[x["label"]]} )
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
accuracy = evaluate.load("accuracy")

f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
      "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
model_name,
num_labels=num_labels
)

args = TrainingArguments(
output_dir="finbert_finetuned",
learning_rate=2e-5,
num_train_epochs=5,
per_device_train_batch_size=16,
per_device_eval_batch_size=32,
eval_strategy="epoch",
save_strategy="epoch",
load_best_model_at_end=True,
metric_for_best_model="f1_macro",
greater_is_better=True,
logging_steps=50,
weight_decay=0.01,
warmup_ratio=0.1,
fp16=True, # if your GPU supports it
report_to="none"
)

trainer = Trainer(
model=model,
args=args,
train_dataset=tokenized["train"],
eval_dataset=tokenized["validation"],
tokenizer=tokenizer,
data_collator=data_collator,
compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print("Validation:", metrics)
test_metrics = trainer.evaluate(tokenized["test"])
print("Test:", test_metrics)

In [ ]:
trainer.save_model("finbert_finetuned_model")
tokenizer.save_pretrained("finbert_finetuned_model")

In [ ]:
model_dir = "finbert_finetuned_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

In [ ]:
pipe2 = TextClassificationPipeline(
model=model,top_k=3,
tokenizer=tokenizer,
return_all_scores=True,
function_to_apply="softmax"
)

In [ ]:
df_test2 = df_test.copy()

In [ ]:
for t in df_test2['text']:
    scores = pipe2(t)[0]
    df_test2.loc[df_test2['text']==t,'Neutral'] = scores[0]['score']
    df_test2.loc[df_test2['text']==t,'Positive'] = scores[1]['score']
    df_test2.loc[df_test2['text']==t,'Negative'] = scores[2]['score']
    top = max(scores, key=lambda x: x["score"])
    df_test2.loc[df_test2['text']==t,'FinBERT'] = top['label'].lower()

In [ ]:
print(accuracy_score(df_test2.label, df_test2.FinBERT))